In [2]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [4]:
def pdf_loader(file_path):
    all_documents = []
    pdf_path = Path(file_path)
    print(pdf_path)
    pdf_files = list(pdf_path.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_path}")
    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}...")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} documents from {pdf_file}")
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents
all_docs = pdf_loader("../data")

../data
Found 3 PDF files in ../data
Processing ../data/pdf/Embeddings.pdf...
Loaded 18 documents from ../data/pdf/Embeddings.pdf
Processing ../data/pdf/Attention.pdf...
Loaded 11 documents from ../data/pdf/Attention.pdf
Processing ../data/pdf/Object_Detection.pdf...
Loaded 22 documents from ../data/pdf/Object_Detection.pdf
Total documents loaded: 51


In [6]:
def chunking(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Total chunks created: {len(split_docs)} from {len(documents)} documents")
    if split_docs:
        print(f"Example chunk content: {split_docs[0].page_content[:200]}...")
        print(f"Example chunk metadata: {split_docs[0].metadata}")
    return split_docs
chunks = chunking(all_docs)

Total chunks created: 289 from 51 documents
Example chunk content: Towards General Text Embeddings with Multi-stage Contrastive Learning
Zehan Li1, Xin Zhang1, Yanzhao Zhang1, Dingkun Long1, Pengjun Xie1, Meishan Zhang
1Alibaba Group
{lizehan.lzh,linzhang.zx,zhangyan...
Example chunk metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-08T01:37:33+00:00', 'source': 'Embeddings.pdf', 'file_path': '../data/pdf/Embeddings.pdf', 'total_pages': 18, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-08-08T01:37:33+00:00', 'trapped': '', 'modDate': 'D:20230808013733Z', 'creationDate': 'D:20230808013733Z', 'page': 0, 'file_type': 'pdf'}


In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb


KeyError: '__reduce_cython__'